In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/orders.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [3]:
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True)

# Verify
df[["Order Date", "Ship Date"]].dtypes

Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object

In [4]:
# Check if any shipping date is earlier than the order date
invalid_dates = df[df["Ship Date"] < df["Order Date"]]
print(f"Invalid shipping dates: {len(invalid_dates)}")

Invalid shipping dates: 0


In [5]:
# First, check for missing values
print(df["Postal Code"].isnull().sum())

# View the rows with missing values
df[df["Postal Code"].isnull()]

11


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
2234,2235,CA-2018-104066,2018-12-05,2018-12-10,Standard Class,QJ-19255,Quincy Jones,Corporate,United States,Burlington,Vermont,NaN,East,TEC-AC-10001013,Technology,Accessories,Logitech ClearChat Comfort/USB Headset H390,205.03
5274,5275,CA-2016-162887,2016-11-07,2016-11-09,Second Class,SV-20785,Stewart Visinsky,Consumer,United States,Burlington,Vermont,NaN,East,FUR-CH-10000595,Furniture,Chairs,Safco Contoured Stacking Chairs,715.20
8798,8799,US-2017-150140,2017-04-06,2017-04-10,Standard Class,VM-21685,Valerie Mitchum,Home Office,United States,Burlington,Vermont,NaN,East,TEC-PH-10002555,Technology,Phones,Nortel Meridian M5316 Digital phone,1294.75
9146,9147,US-2017-165505,2017-01-23,2017-01-27,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,Vermont,NaN,East,TEC-AC-10002926,Technology,Accessories,Logitech Wireless Marathon Mouse M705,99.98
9147,9148,US-2017-165505,2017-01-23,2017-01-27,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,Vermont,NaN,East,OFF-AR-10003477,Office Supplies,Art,4009 Highlighters,8.04
9148,9149,US-2017-165505,2017-01-23,2017-01-27,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,Vermont,NaN,East,OFF-ST-10001526,Office Supplies,Storage,Iceberg Mobile Mega Data/Printer Cart,1564.29
9386,9387,US-2018-127292,2018-01-19,2018-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-PA-10000157,Office Supplies,Paper,Xerox 191,79.92
9387,9388,US-2018-127292,2018-01-19,2018-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-PA-10001970,Office Supplies,Paper,Xerox 1881,12.28
9388,9389,US-2018-127292,2018-01-19,2018-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-AP-10000828,Office Supplies,Appliances,Avanti 4.4 Cu. Ft. Refrigerator,542.94
9389,9390,US-2018-127292,2018-01-19,2018-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-EN-10001509,Office Supplies,Envelopes,Poly String Tie Envelopes,2.04


In [6]:
# Fill missing postal codes with "Unknown"
df["Postal Code"] = df["Postal Code"].fillna("Unknown")

In [ ]:
df["Postal Code"] = pd.to_numeric(
    df["Postal Code"],
    errors="coerce"
).astype("Int64")

In [7]:
# Check for duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


In [8]:
df.columns = df.columns.str.strip()  # Remove extra spaces
df.columns = df.columns.str.replace(" ", "_")  # Replace spaces with underscores (optional, depending on the client)

print(df.columns)

Index(['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode',
       'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State',
       'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub-Category',
       'Product_Name', 'Sales'],
      dtype='str')


In [10]:
# Sales negative to nahi
print("Negative Sales:", (df["Sales"] < 0).sum())


Negative Sales: 0


In [9]:
# Check Sales distribution
df["Sales"].describe()

# Detect outliers using the IQR method
Q1 = df["Sales"].quantile(0.25)
Q3 = df["Sales"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["Sales"] < lower_bound) |
    (df["Sales"] > upper_bound)
]

print(f"Outliers in Sales: {len(outliers)}")

Outliers in Sales: 1145


In [11]:
df.head(2)

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94


In [11]:
# Check karo ek Customer ID hamesha same Customer Name se match karta hai
df.groupby("Customer_ID")["Customer_Name"].nunique().sort_values(ascending=False).head()

Customer_ID
AA-10315    1
AA-10375    1
AA-10480    1
AA-10645    1
AB-10015    1
Name: Customer_Name, dtype: int64

In [12]:
# Check whether each State belongs to only one Region
df.groupby("State")["Region"].nunique().sort_values(ascending=False).head()

State
Alabama       1
Arizona       1
Arkansas      1
California    1
Colorado      1
Name: Region, dtype: int64

In [14]:
df.info()
df.duplicated().sum()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row_ID         9800 non-null   int64         
 1   Order_ID       9800 non-null   str           
 2   Order_Date     9800 non-null   datetime64[us]
 3   Ship_Date      9800 non-null   datetime64[us]
 4   Ship_Mode      9800 non-null   str           
 5   Customer_ID    9800 non-null   str           
 6   Customer_Name  9800 non-null   str           
 7   Segment        9800 non-null   str           
 8   Country        9800 non-null   str           
 9   City           9800 non-null   str           
 10  State          9800 non-null   str           
 11  Postal_Code    9789 non-null   Int64         
 12  Region         9800 non-null   str           
 13  Product_ID     9800 non-null   str           
 14  Category       9800 non-null   str           
 15  Sub-Category   9800 non-null   s

,Row_ID,Order_Date,Ship_Date,Postal_Code,Sales
count,9800.000000,9800,9800,9789.0,9800.000000
mean,4900.500000,2017-05-01 05:13:51.673469,2017-05-05 04:17:52.653061,55273.322403,230.769059
min,1.000000,2015-01-03 00:00:00,2015-01-07 00:00:00,1040.0,0.444000
25%,2450.750000,2016-05-24 00:00:00,2016-05-27 18:00:00,23223.0,17.248000
50%,4900.500000,2017-06-26 00:00:00,2017-06-29 00:00:00,58103.0,54.490000
75%,7350.250000,2018-05-15 00:00:00,2018-05-19 00:00:00,90008.0,210.605000
max,9800.000000,2018-12-30 00:00:00,2019-01-05 00:00:00,99301.0,22638.480000
std,2829.160653,NaN,NaN,32041.223413,626.651875


In [13]:
import os

# Create the processed data folder if it does not already exist
os.makedirs("../data/processed", exist_ok=True)

# Save the cleaned dataset as a CSV file
df.to_csv("../data/processed/cleaned_orders.csv", index=False)
print("Cleaned data saved successfully!")

Cleaned data saved successfully!


## 🧹 Data Cleaning Summary

### Actions Taken:
- Converted `Order Date` and `Ship Date` to datetime format
- Converted `Postal Code` to nullable integer type
- Handled 11 missing values in `Postal Code` (0.11%) without removing rows
- Checked for duplicate rows
- Cleaned column names by removing extra spaces
- Verified no negative values in `Sales`
- Standardized text in categorical columns

### Final Dataset:
- Total Rows: 9,800
- Total Columns: 18
- Saved to: `data/processed/cleaned_orders.csv`

### Ready for:
- Sales Performance Analysis (Notebook 03)